In [6]:
import librosa
import os
import numpy as np
import soundfile as sf
import pandas as pd
from tqdm import tqdm



In [7]:
def check_audio(path, target_sr= 16000):
    result = {
        "file": path,
        "status": "PASS",
        "sample_rate": "None",
        "channels": "None",
        "duration": "None",
        "clipping_ratio": "None",
        "silence_ratio": "None",
        "reason": ""
    }

    try:
        info = sf.info(path)
        result["sample_rate"] = info.samplerate
        result["channels"]= info.channels
        result["duration"]= info.duration

        y, sr= librosa.load(
            path,
            sr= target_sr,
            mono= True
        )

        if not np.all(np.isfinite):
            result["status"] = "REJECT"
            result["reason"] = "Nan_or_Inf"
            return result

        if not 2.0 <= len(y) / sr <= 5.0:
            result["status"] = "REJECT"
            result["reason"] = "Nan_or_Inf"
            return result

        clipping_ratio= np.mean(np.abs(y) >= 0.99)
        result["clipping_ratio"] = clipping_ratio

        if clipping_ratio > 0.01:
            result["status"] = "REJECT"
            result["reason"] = "too_much_clipping"
            return result

        intervals = librosa.effects.split(
            y,
            top_db=40
        )

        if len(intervals) == 0:
            result["status"] = "REJECT"
            result["reason"] = "Complete_silence"
            return result

        voiced_samples= sum(
            end - start
            for start, end in intervals
        )

        silence_ratio= 1 - voiced_samples / len(y)
        result["silence_ratio"]= silence_ratio

        if silence_ratio > 0.70:
            result["status"] = "REJECT"
            result["reason"] = "too_much_silence"


    except Exception as e:
        result["status"]= "Reject"
        result["reason"]= f"Error: {str(e)}"

    return result

In [18]:
dir= r"D:\datas_suara_split\Record\val"

results= []

for root, dirs, files in os.walk(dir):
    for file in tqdm(files):
        if file.lower().endswith(".wav"):
            path= os.path.join(root, file)

            result = check_audio(path)
            results.append(result)
df= pd.DataFrame(results)

df.to_csv(
    "audio_qc.csv",
    index= False

)

print(df["status"].value_counts())


100%|██████████| 150/150 [00:02<00:00, 53.15it/s]

status
PASS    150
Name: count, dtype: int64
